# Buổi 24 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `ensemble.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Dữ liệu và 30 ứng viên (mục 4)

Lần đầu 3–6 phút (22 mô hình thống kê + 2 LightGBM × 3 cửa sổ), lưu vào `lab/du-lieu/cache/`.

In [ ]:
%matplotlib inline
import warnings

import cold_start as cs
import ensemble as en

warnings.simplefilter("ignore")
df = en.doc_m4()
fc = en.du_bao_30(df, luu=True)
diem = en.mase(fc, df)                          # MASE: mỗi dòng một (chuỗi, cutoff), mỗi cột một ứng viên
print("số chuỗi:", df["unique_id"].nunique(), "| số ứng viên:", len(en.ung_vien(fc)))
diem.groupby("cutoff").mean().T.sort_values(131).round(3)

## Bước 2 — Ensemble định sẵn so với mô hình đơn (mục 4.1)

In [ ]:
w3 = diem.xs(131, level="cutoff")              # cửa sổ W3
for m in ["SeasonalNaive", "AutoETS", "AutoTheta", "LGBM", "TB(ETS,Theta)", "TB(ETS,Theta,LGBM)"]:
    print(f"{m:20s} MASE W3 {w3[m].mean():.3f}")

## Bước 3 — Chọn mô hình cho từng chuỗi (mục 4.3)

Sửa `CUA_SO_CHON`, `CUA_SO_BAO_CAO` và `chon_va_bao_cao` trong `ensemble.py` rồi chạy lại hai ô dưới.

In [ ]:
print("chọn trên", en.CUA_SO_CHON, "| báo cáo trên", en.CUA_SO_BAO_CAO)
for theo_chuoi in (True, False):
    kq = en.chon_va_bao_cao(diem, theo_chuoi=theo_chuoi)
    print("theo từng chuỗi" if theo_chuoi else "một lựa chọn chung",
          {k: round(v, 3) for k, v in kq.items() if k != "lua_chon"})

In [ ]:
lq = en.lac_quan_theo_so_ung_vien(diem)      # khoảng 30 giây
ax = lq[["diem_luc_chon", "diem_bao_cao"]].plot(marker="o")
ax.set_xlabel("số ứng viên được thử cho mỗi chuỗi")
ax.set_ylabel("MASE trung bình")
lq.round(3)

## Bước 4 — Trọng số (mục 4.2)

In [ ]:
for cach in ["deu", "nghich_mse", "toi_uu"]:
    w = en.trong_so(fc, cach)                   # ước lượng trên en.CUA_SO_CHON
    f3 = en.ap_trong_so(fc[fc["cutoff"] == 131], w, "ket_hop")
    m = en.mase(f3[["unique_id", "ds", "cutoff", "y", "ket_hop"]], df)["ket_hop"].mean()
    print(f"{cach:11s} MASE W3 {m:.3f} | trọng số lớn nhất {w.to_numpy().max():.2f}")

## Bước 5 — AutoGluon (mục 4.4)

`medium_quality`, giới hạn 300 giây, bỏ Chronos-2 và Toto-2. Khoảng 1–3 phút.

In [ ]:
fa, bxh, trong = en.chay_autogluon(df, time_limit=300)
fa = fa.merge(df[["unique_id", "ds", "y"]], on=["unique_id", "ds"])
bxh = bxh.assign(mase_w3=bxh["model"].map(en.mase(fa, df).mean()))
print("trọng số ensemble:", {k: round(v, 3) for k, v in trong.items()})
bxh.round(3)

## Bước 6 — Cold start (mục 4.5)

Lần đầu đọc tệp Excel mất 1–2 phút.

In [ ]:
sp, ban = cs.doc_san_pham()
moi, chon = cs.chia_tap(sp, ban)
k = cs.chon_k(sp, ban, chon)                    # chọn k trên các mã ra mắt TRƯỚC tập 200 mã
print(len(moi), "mã mới, ra mắt", moi["ra_mat"].min().date(), "→", moi["ra_mat"].max().date(), "| k =", k)
for cach in ["tb_danh_muc", "analog_danh_muc", "analog_gia"]:
    print(f"{cach:16s}", {t: round(v, 3) for t, v in cs.danh_gia(sp, ban, moi, cach, k).items()})

## Bước 7 — Kiểm tra

Trong terminal ở thư mục `lab/`: `python lab.py check` — phải xanh 6/6.